In [1]:
!curl -L "https://raw.githubusercontent.com/aymanya/Arabic-Sentiment-Analysis-Datasets/main/ArabicRealstateDataset.csv" -o ArabicRealstateDataset.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1042k  100 1042k    0     0  1980k      0 --:--:-- --:--:-- --:--:-- 1985k


In [2]:
import pandas as pd

df = pd.read_csv("ArabicRealstateDataset.csv")
print(df.shape)
print(df.columns.tolist())
df.head(10)

(6434, 2)
['Text', 'Polarity']


,Text,Polarity
0,"2000 ريال للمتر, من المغفل اللي بيشتري بهالسعر.",Neg
1,الله ينتقم على كل عقاري يستغل المواطن برفع الا...,Neg
2,قرض يقص الظهر و المستفيد الوحيد البنوك.,Neg
3,غلاء فاحش فالعقار ولا يستاهل هالقيمة الغالية و...,Neg
4,وزارة الاسكان و الصندوق العقاري يساعد على الفس...,Neg
5,نريد القرض العقاري 500 الف بدون فوائد غيرها بل...,Neg
6,خبر طيب لو تم عاجلا بدون مقدمات لأهل العقار ، ...,Pos
7,القسط أقوى من النظام القديم طبعا الأنها الية ب...,Mix
8,وزارة الاسكان و الصندوق العقاري دعم لا محدود ل...,Neg
9,مع وزارة الاسكان تنتقل من هم السكن الى هم الدي...,Neg


Remarks:
- 2 cols: text (ar), and polarity (neg / pos/mix)
- text is non-formal at times mixed in with numbers and currency.
- some rows have commas inside the text so this requires some encoding and parsing attention

When it comes to the pipeline possible LLM task here could be:
- extracting the topic being discussed (loan interest, property prices...etc)
- extracting any numbers (prices, dimensions...etc)
- generating a general sentinment and comparing against the "ground truth" label


In [3]:
# shape and null check
print(df.shape)
print(df.isnull().sum())

# polarity distribution
print(df['Polarity'].value_counts())

# text length distribution
df['text_length'] = df['Text'].str.len()
print(df['text_length'].describe())

(6434, 2)
Text        0
Polarity    0
dtype: int64
Polarity
Pos    2916
Neg    2914
Mix     604
Name: count, dtype: int64
count    6434.000000
mean       89.124495
std        49.996172
min        16.000000
25%        50.000000
50%        76.000000
75%       118.000000
max       243.000000
Name: text_length, dtype: float64


- dataset clean so no missing data to handle
- also pretty balanced in terms of polarity but mix (604) is the minority class
- general comments are short in length so its gonna be better for the LLM processing for my constraints

In [4]:
# encoding and any non-Arabic characters
print(df['Text'].head(3).tolist())

# duplicates
print(f"Duplicates: {df.duplicated(subset='Text').sum()}")

#  polarity values are clean?
print(df['Polarity'].unique())

# sampling a Mix row specifically
print(df[df['Polarity'] == 'Mix'].sample(3)['Text'].tolist())

[' 2000 ريال للمتر, من المغفل اللي بيشتري بهالسعر.', 'الله ينتقم على كل عقاري يستغل المواطن برفع الاسعار مقابل حاجة لمسكن و يجعل ما يربحه يصرفه على صحته.', 'قرض يقص الظهر و المستفيد الوحيد البنوك.']
Duplicates: 0
['Neg' 'Pos' 'Mix']
['لا يوجد نزول داخل المدينة. النزول على أطراف المدينة.', 'هذه الزيادة السعرية الكبيرة سوف تجعل الهبوط السعري القادم قاسي.', 'أمس كنت فى أبحر بعد غياب شهرين حقيقة تفاجأت بالنمو العمرانى بس نفسى أعرف شركة الصرف الصحى ليش عملهم بطئ.']
